In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_unstructured import UnstructuredLoader
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_huggingface import HuggingFaceEmbeddings
import glob
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from typing import TypedDict, List, Literal
import re
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langgraph.checkpoint.memory import MemorySaver
from langchain_tavily import TavilySearch
import glob
from langgraph.types import interrupt, Command

In [ ]:
#file path
files = glob.glob("documents/**/*", recursive=True)
files

In [ ]:
loader = UnstructuredLoader(files,chunking_strategy="by_title")    

In [ ]:
docs = loader.load()

In [ ]:
#splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
chunks = splitter.split_documents(docs)

In [ ]:
#embedding models
# embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"batch_size": 64, "normalize_embeddings": True}
)

In [ ]:
#rertriever
retriever = FAISS.from_documents(chunks, embedding_model).as_retriever(search_type='similarity', search_kwargs={'k':4})

In [ ]:
#model
llm  = ChatGroq(model="openai/gpt-oss-20b")

# 1. retrieval Decision : whether it requires external docs or it can answer in its own

In [ ]:
class State(TypedDict):
    question: str
    docs: List[Document]
    answer: str
    #step 1: creating decision state to check if retrieval is needed
    need_retrieval: bool
    

In [ ]:
#for structured output
class retrievalDecision(BaseModel):
    decision : bool

In [ ]:
decide_retrieval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Decide whether retrieval is needed to answer the question. "
            "Use the retrievalDecision tool and return only the structured result. "
            "Set decision=True if answering requires specific facts, citations, "
            "or information likely not in the model. Set decision=False for general "
            "explanations, definitions, or reasoning that does not need sources. "
            "If unsure, choose True."
        ),
        ("human", "Question: {question}"),
    ]
)

should_retrieve_llm = llm.with_structured_output(
    retrievalDecision)

In [ ]:
def decide_retrieval(state: State):
    decision = should_retrieve_llm.invoke(decide_retrieval_prompt.format(question=state["question"]))
    return {"need_retrieval": decision.decision}

In [ ]:
# generate direct : which will not retrieve any documents
def generate_direct(state: State):
    answer = llm.invoke(
        [
            (
                "system",
                "You are a helpful assistant. Answer the question based on your knowledge."
            ),
            ("human", f"Question: {state['question']}"),
        ]
    )
    return {"answer": answer.content}


In [ ]:
def retrieve(state: State):
    return {"docs": retriever.invoke(state["question"])}


In [ ]:
# function for deciding whether to retrieve or not
def whether_retrieve(state: State):
    if state['need_retrieval']:
        return 'retrieve'
    else :
        return 'generate_direct'

In [ ]:
#build graph
graph  = StateGraph(State)

#adding nodes
graph.add_node('decide_retrieval', decide_retrieval)
graph.add_node('generate_direct', generate_direct)
graph.add_node('retrieve', retrieve)

#adding edges
graph.add_edge(START, 'decide_retrieval')
graph.add_conditional_edges('decide_retrieval', whether_retrieve,{'retrieve': 'retrieve', 'generate_direct': 'generate_direct'})

graph.add_edge('generate_direct',END)
graph.add_edge('retrieve',END)

#compile
app = graph.compile()
app

In [ ]:
result = app.invoke(
    {
        "question": 'who is the CEO of Nexa AI ?'
    }
)

print(result["need_retrieval"])

# 2. Creating relevant_docs as key in our State in case our model requires retrieval

In [ ]:
# Graph State
# --------------------------------------------------
class State(TypedDict):
    question: str
    docs: List[Document]
    answer: str
    need_retrieval: bool

    # step 2 : adding relevant_docs
    relevant_docs: List[Document]

In [ ]:
# schema for judging docs
class judgeDocs(BaseModel):
    is_relevant : bool = Field(..., description="True if the document is relevant to the question, False otherwise.")

In [ ]:
is_relevant_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are judging document relevance. "
            "Return only the provided structured output. "
            "A document is relevant if it contains information useful for answering the question."
        ),
        (
            "human",
            "Question:\n{question}\n\nDocument:\n{document}"
        ),
    ]
)

#llm with relevance judgement output
relevance_judgement_llm = llm.with_structured_output(judgeDocs)

In [ ]:
#now creating node for judging docs
def is_relevant(state: State):
    question = state["question"]
    docs = state['docs']
    #empty
    relevant_docs : List[Document] = []

    for doc in docs:
        relevant = relevance_judgement_llm.invoke(is_relevant_prompt.format(question=question, document=doc.page_content))
        if relevant.is_relevant:
            relevant_docs.append(doc)

    return {"relevant_docs": relevant_docs}

In [ ]:
graph  = StateGraph(State)

#adding nodes
graph.add_node('decide_retrieval', decide_retrieval)
graph.add_node('generate_direct', generate_direct)
graph.add_node('retrieve', retrieve)
graph.add_node('is_relevant', is_relevant)

#adding edges
graph.add_edge(START, 'decide_retrieval')
graph.add_conditional_edges('decide_retrieval', whether_retrieve,{'retrieve': 'retrieve', 'generate_direct': 'generate_direct'})

graph.add_edge('retrieve','is_relevant')
graph.add_edge('is_relevant',END)
graph.add_edge('generate_direct',END)


#compile
app = graph.compile()
app

In [ ]:
result = app.invoke(
    {
        "question": 'who is the CEO of NexaAI ?'
    }
)

print(result["need_retrieval"])

In [ ]:
#check relevant docs
print('\n'.join([i.page_content for i in  result['relevant_docs']]))

# 3. Generate from relevant docs

`Generating from relevant documents if available otherwise return "No relevant information found"`

In [ ]:
# creating node : generate from context
def generate_from_context(state:State):
    #get question and relevant docs from state
    q = state['question']
    context = '\n'.join([i.page_content for i in state['relevant_docs']])

    #prompt
    prompt = ChatPromptTemplate.from_messages([
        ('system',
         "You are a helpful assistant, answer the following question using only provided context"),
         ('human',
          'Question : {question}\n\n context : {context}')
    ])
    # chain
    generate_from_context_chain  = prompt | llm

    #response
    response = generate_from_context_chain.invoke({'question':q, 'context':context})
    return {'answer': response.content}

In [ ]:
# creating node called : no_relevant_docs
def no_relevant_docs(state:State):
    return {'answer':'No relevant information'}

In [ ]:
# creating conditional edge function
def docsORnodocs(state:State):
    if state['relevant_docs']:
        return 'generate_from_context'
    else :
        return 'no_relevant_docs'

In [ ]:
# build graph now
graph  = StateGraph(State)

#adding nodes
graph.add_node('decide_retrieval', decide_retrieval)
graph.add_node('generate_direct', generate_direct)
graph.add_node('retrieve', retrieve)
graph.add_node('is_relevant', is_relevant)
graph.add_node('generate_from_context', generate_from_context)
graph.add_node('no_relevant_docs', no_relevant_docs)

#adding edges
graph.add_edge(START, 'decide_retrieval')
graph.add_conditional_edges('decide_retrieval', whether_retrieve,{'retrieve': 'retrieve', 'generate_direct': 'generate_direct'})

graph.add_edge('retrieve','is_relevant')
#step 3 is adding
graph.add_conditional_edges('is_relevant',docsORnodocs,{'generate_from_context': 'generate_from_context', 'no_relevant_docs': 'no_relevant_docs'})
graph.add_edge("generate_from_context", END)
graph.add_edge("no_relevant_docs", END)
#------------------------------------------

graph.add_edge('generate_direct',END)


#compile
app = graph.compile()
app

In [ ]:
result = app.invoke(
    {
        "question": 'who is the CEO of NexaAI ?'
    }
)

print(result["answer"])

In [ ]:
result = app.invoke(
    {
        "question": 'who is CEO of apple in 2026?'
    }
)

print(result["answer"])

# 4. Creating "is_sup" node after generating_from_context

`which will check if our generated context is sufficient to answer the question also it will`

In [ ]:
class State(TypedDict):
    question: str
    docs: List[Document]
    answer: str
    need_retrieval: bool
    relevant_docs: List[Document]

    #step 4
    is_hallucinating : bool
    is_supporting : str

In [ ]:
# creating pydantic schema
class is_suff_schema(BaseModel):
    answer : Literal['fully_supported', 'partially_supported', 'no_supported']
    is_hallucinating : bool = Field(description="Whether the answer is hallucinating or not")
    

In [ ]:
is_suff_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a question and answer supervisor. Evaluate how well the generated answer is supported by the provided context and be fair.\n\n'
     'Return one of these values for the "answer" field:\n'
     '- fully_supported: when the context contains all information needed to answer the question and the generated answer is accurate based on the context\n'
     '- partially_supported: when the context has some relevant information but is incomplete or the answer has some unsupported details\n'
     '- no_supported: when the context does not contain information to answer the question or the answer contradicts the context\n\n'
     'For is_hallucinating:\n'
     '- True: if the answer contains information not present in the context or makes claims unsupported by the context\n'
     '- False: if the answer is fully derived from the provided context'),
     ('human',
      "Question: {question}\n\nContext: {context}\n\nGenerated Answer: {answer}")
])

#llm with structured output
is_suff_llm = is_suff_prompt | llm.with_structured_output(is_suff_schema)

In [ ]:
# def creating is_suff node
def is_suff(state:State):
    question = state['question']
    answer = state['answer']
    context = '\n'.join([i.page_content for i in state['relevant_docs']])

    #output
    res = is_suff_llm.invoke({'question':question,'answer':answer, 'context':context})

    return {'is_supporting': res.answer,
            'is_hallucinating': res.is_hallucinating }

In [ ]:
# build graph now
graph  = StateGraph(State)

#adding nodes
graph.add_node('decide_retrieval', decide_retrieval)
graph.add_node('generate_direct', generate_direct)
graph.add_node('retrieve', retrieve)
graph.add_node('is_relevant', is_relevant)
graph.add_node('generate_from_context', generate_from_context)
graph.add_node('no_relevant_docs', no_relevant_docs)
graph.add_node('is_suff',is_suff)

#adding edges
graph.add_edge(START, 'decide_retrieval')
graph.add_conditional_edges('decide_retrieval', whether_retrieve,{'retrieve': 'retrieve', 'generate_direct': 'generate_direct'})

graph.add_edge('retrieve','is_relevant')
#step 3 is adding
graph.add_conditional_edges('is_relevant',docsORnodocs,{'generate_from_context': 'generate_from_context', 'no_relevant_docs': 'no_relevant_docs'})
#step 4 : is_sufficient
graph.add_edge("generate_from_context", 'is_suff')
graph.add_edge('is_suff', END)
graph.add_edge("no_relevant_docs", END)
#------------------------------------------

graph.add_edge('generate_direct',END)


#compile
app = graph.compile()
app

In [ ]:
result = app.invoke(
    {
        "question": 'who is the CEO of NexaAI ?'
    }
)

print("Answer:", result["answer"])
print("is_supporting:", result["is_supporting"])
print("is_hallucinating:", result["is_hallucinating"])

In [ ]:
result = app.invoke(
    {
        "question": "Describe NexaAI's company culture"
    }
)

print("Answer:", result["answer"])
print("is_supporting:", result["is_supporting"])
print("is_hallucinating:", result["is_hallucinating"])

# 5. Adding Web Search Node

`adding web_search node which will search on internet about the question`

In [ ]:
class State(TypedDict):
    question: str
    docs: List[Document]
    answer: str
    need_retrieval: bool
    relevant_docs: List[Document]
    is_hallucinating : bool
    is_supporting : str

    #step 5 : web search
    web_searched : bool
    refined_query : str

In [ ]:
#we also need to make a query refiner which will refine user's query before searching on the web so we get more accurate results
class queryRefiner(BaseModel):
    refined_query : str

query_prompt = ChatPromptTemplate.from_messages([(
    'system',
    "You are a query refiner. Refine the user query to make it more specific and relevant for web search.\n"),(
        'human',
        "Question : {question}"
)])

In [ ]:
# web Search Node
def webSearch(state : State):
    question = state['question']

    #refining the query to get more relevant results
    refined_query = (query_prompt | llm.with_structured_output(queryRefiner)).invoke({'question': question}).refined_query

    #searching the web using TavilySearch
    web  = TavilySearch()
    web_res = web.invoke(refined_query, max_results=3)
    return {'docs': [Document(i['content']) for i in web_res['results']],
            'web_searched' : True,
            'refined_query': refined_query}

In [ ]:
# build graph now
graph  = StateGraph(State)

#adding nodes
graph.add_node('decide_retrieval', decide_retrieval)
graph.add_node('generate_direct', generate_direct)
graph.add_node('retrieve', retrieve)
graph.add_node('is_relevant', is_relevant)
graph.add_node('generate_from_context', generate_from_context)
graph.add_node('no_relevant_docs', no_relevant_docs)
graph.add_node('is_suff',is_suff)
graph.add_node('webSearch', webSearch)

#adding edges
graph.add_edge(START, 'decide_retrieval')
graph.add_conditional_edges('decide_retrieval', whether_retrieve,{'retrieve': 'retrieve', 'generate_direct': 'generate_direct'})

graph.add_edge('retrieve','is_relevant')
#step 3 is adding
graph.add_conditional_edges('is_relevant',docsORnodocs,{'generate_from_context': 'generate_from_context', 'no_relevant_docs': 'no_relevant_docs'})
#step 4 : is_sufficient
graph.add_edge("generate_from_context", 'is_suff')
graph.add_edge('is_suff', END)
#step 5: web search after no_relevant_docs
graph.add_edge("no_relevant_docs", 'webSearch')
graph.add_edge('webSearch', 'is_relevant')
graph.add_edge('generate_direct',END)


#compile
app = graph.compile()
app

In [ ]:
result = app.invoke(
    {
        "question": "why real madrid's last matches were disaster? Why the club is performing very poor? "
    }
)

print("Answer:", result["answer"])

In [ ]:
result.keys()

In [ ]:
print(f"Question :{result['question']}\n                                    vs\nRefined Question : {result['refined_query']}")

# 6. Adding HITL before webSearch Node

`adding HITL before moving to webSearch Node`

In [ ]:
class State(TypedDict):
    question: str
    docs: List[Document]
    answer: str
    need_retrieval: bool
    relevant_docs: List[Document]
    is_hallucinating : bool
    is_supporting : str

    #step 5 : web search
    web_search : bool
    refined_query : str

In [ ]:
# modifying node called : no_relevant_docs :adding HITL
def no_relevant_docs(state:State):
    #ask user
    answer = interrupt('there is no releveant information about the query you asked. Do you want to search on web ? - yes/no')

    if str(answer).lower().strip().startswith('yes'):
        return {'web_search' : True, 'answer':'No relevant information'}
    else :
        return {'web_search' : False, 'answer':'No relevant information'}


In [ ]:
# modifying webSearch Node
def webSearch(state : State):
    question = state['question']

    #refining the query to get more relevant results
    refined_query = (query_prompt | llm.with_structured_output(queryRefiner)).invoke({'question': question}).refined_query

    #searching the web using TavilySearch
    web  = TavilySearch()
    web_res = web.invoke(refined_query, max_results=3)
    return {'docs': [Document(i['content']) for i in web_res['results']],
            # 'web_searched' : True,                  passing it in no_relevant_docs now
            'refined_query': refined_query}

In [ ]:
#function for conditional fucntion
def routeDecide(state:State):
    if state['web_search']:
        return 'search_on_web'
    else :
        return 'no_need'

In [ ]:
# build graph now
graph  = StateGraph(State)

#adding nodes
graph.add_node('decide_retrieval', decide_retrieval)
graph.add_node('generate_direct', generate_direct)
graph.add_node('retrieve', retrieve)
graph.add_node('is_relevant', is_relevant)
graph.add_node('generate_from_context', generate_from_context)
graph.add_node('no_relevant_docs', no_relevant_docs)
graph.add_node('is_suff',is_suff)
graph.add_node('webSearch', webSearch)

#adding edges
graph.add_edge(START, 'decide_retrieval')
graph.add_conditional_edges('decide_retrieval', whether_retrieve,{'retrieve': 'retrieve', 'generate_direct': 'generate_direct'})

graph.add_edge('retrieve','is_relevant')
#step 3 is adding
graph.add_conditional_edges('is_relevant',docsORnodocs,{'generate_from_context': 'generate_from_context', 'no_relevant_docs': 'no_relevant_docs'})
#step 4 : is_sufficient
graph.add_edge("generate_from_context", 'is_suff')
graph.add_edge('is_suff', END)
#step 5: web search after no_relevant_docs
graph.add_conditional_edges("no_relevant_docs", routeDecide, {'search_on_web':'webSearch', 'no_need': END} )
graph.add_edge('webSearch', 'is_relevant')
graph.add_edge('generate_direct',END)


#compile
app = graph.compile(checkpointer=MemorySaver())
app

In [ ]:
config = {"configurable": {"thread_id": "2"}}

In [ ]:
result = app.invoke(
    {
        "question": "tell me about the financial performance about NexaAi company"
    }, config=config
)

if result.get('__interrupt__', ''):
    print(result.get('__interrupt__', '')[-1].value)
else :
    print(result['answer'])

In [ ]:
result = app.invoke(
    Command(resume = 'yes'),
    config=config
)
if result.get('__interrupt__', ''):
    print(result.get('__interrupt__', '')[-1].value)
else :
    print(result['answer'])

In [ ]:
result.keys()

In [ ]:
result['web_search']